# Colorado snowpack (SWE) — liberation pipeline

Thin orchestration over the tested `snowpack` package. Runs four stages:
**retrieve → audit → cleanup → publish**. The package holds all the logic; this
notebook is the human-facing narrative. Implements issue [#11](https://github.com/CUPIDS-Lab/co-environmental-data/issues/11);
stamped out from the sibling `streamflow` pipeline (#10).

**⚠️ Two networks that complement, not overlap:** SNOTEL (automated, daily, ~1980+)
and snow courses (manual, semimonthly, back to the 1930s–40s) are *nearby but
distinct* sites — keep them separate, don't average them. The headline product is
**basin SWE as a percent of normal** (`audit.basin_percent_normal`). The AWDB API
needs **no key**. See `docs/survey-notes.md` and `data/lookups/concepts.yaml`.

In [ ]:
# Config
MODE = "live"          # "live" = fetch the AWDB API | "demo" = offline 1-station sample
FRESH = True           # True clears data/original first so the CSV reflects exactly this run
SOURCES = None         # None = both | ["nrcs_snotel"] | ["nrcs_snowcourse"]
# Sampling hook: station triplets (or bare ids). Set SITES = None for the full
# seed (199 stations × full POR). Here: the co-located Park Cone pair + one more.
SITES = {"680:CO:SNTL", "06L02:CO:SNOW", "713:CO:SNTL"}

from snowpack import fetch, audit, clean, config
import pandas as pd
print("sources:", list(config.get_sources()))

## 1. Retrieve — fetch raw responses → immutable `data/original/` + manifest

`fetch_all` is idempotent (skips cached files), resilient (no-data and transient
errors are durable, not fatal), and reports progress. AWDB returns each station's
full period of record in one response and needs **no API key**.

In [ ]:
import shutil
if FRESH and config.ORIGINAL.exists():
    shutil.rmtree(config.ORIGINAL)
config.ORIGINAL.mkdir(parents=True, exist_ok=True)

if MODE == "live":
    artifacts = fetch.fetch_all(sources=SOURCES, sites=SITES)
    print(f"fetched {len(artifacts)} stations")
else:
    # offline demo: seed one SNOTEL fixture (Park Cone)
    src = config.get_sources()["nrcs_snotel"]
    art = next(a for a in src.discover(sites={"680:CO:SNTL"}))
    art.local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(config.PROJECT_DIR / "tests" / "fixtures" / "nrcs_snotel_sample.json", art.local_path)
    print("demo seed written")

## 2. Audit (raw) — did the fetch return data?

In [ ]:
print(audit.profile_raw())

## 3. Cleanup — parse → tidy long → validate → `data/processed/snowpack.csv`

In [ ]:
data, prov = clean.run(sources=SOURCES, sites=SITES, fail_on_empty=True)
print(f"{len(data):,} rows · {data.site_id.nunique()} stations · sources={sorted(data.source.unique())}")
data.head()

## 4. Audit (processed) — profile, coverage, basin % of normal, cross-source

`basin_percent_normal` is the signature snowpack product (basin SWE vs the
1991–2020 day-of-year median). `reconcile_cross_source` checks co-located
SNOTEL↔snow-course pairs — a *looser* agreement than streamflow's re-serve, since
they are nearby-not-identical sites. (% of normal is meaningful in snow season.)

In [ ]:
print(audit.audit_processed())
cov = audit.coverage_report()
audit.variables_report()

# Basin % of normal — pick an in-season date (snow season ~Nov–Jun) for a meaningful number.
bpn = audit.basin_percent_normal(target_date="2024-04-01")
print("\nBasin SWE % of normal (as of 2024-04-01):")
display(bpn)

xs = audit.reconcile_cross_source()
print("\nCross-source agreement (SNOTEL vs co-located snow course):")
xs

## 5. Publish — finalize the deliverable

The processed CSV is the deliverable. Optionally reconcile basin % of normal
against the NRCS basin report (`audit.reconcile_basin_pct_normal`), then deposit to
Dataverse for a DOI (`dataverse/DEPOSIT.md`). The headless twin
`python -m snowpack.pipeline` runs all of the above for CI.

In [ ]:
# Optional spot-check vs the NRCS Colorado Snow Survey basin report (% of normal):
expected = {
    # "Gunnison": 106,   # read off the NRCS basin report for the run date, then uncomment
}
if expected:
    display(audit.reconcile_basin_pct_normal(expected))
print("deliverable:", config.CANONICAL_CSV)